# CNN Explainability with Grad-CAM

This notebook plots a heatmap showing which image regions the CNN considers important for its classification.

- Update the paths in the example cell (`MODEL_PATH`, `IMAGE_PATH`) to match your file.
- The script auto-selects the final convolutional layer for Grad-CAM.
- It displays both the prediction and the Grad-CAM overlay.

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt


class CNNBinary(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.fwd_handle = self.target_layer.register_forward_hook(self._save_activations)

    def _save_activations(self, module, inp, out):
        self.activations = out

    def _save_activation_grads(self, grad):
        self.gradients = grad.detach()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        self.gradients = None
        self.activations = None

        logits = self.model(x)

        if self.activations is None:
            raise RuntimeError("Grad-CAM failed to capture activations from the target layer.")

        if self.activations.requires_grad:
            self.activations.register_hook(self._save_activation_grads)
        else:
            raise RuntimeError("Target layer activations do not require gradients.")

        if logits.ndim == 1:
            logits = logits.unsqueeze(0)

        if logits.shape[1] == 1:
            # Binary case: one logit, use sigmoid to get crack probability
            crack_prob = torch.sigmoid(logits[:, 0])
            pred_idx = int((crack_prob >= 0.5).item())
            probs = np.array([1.0 - float(crack_prob.item()), float(crack_prob.item())], dtype=np.float32)

            if class_idx is None:
                class_idx = pred_idx

            # class_idx: 0 = Non-Cracked, 1 = Cracked
            score = logits[:, 0].sum() if class_idx == 1 else (-logits[:, 0]).sum()
        else:
            probs_t = torch.softmax(logits, dim=1)
            pred_idx = int(torch.argmax(probs_t, dim=1).item())
            probs = probs_t.detach().cpu().numpy()[0]

            if class_idx is None:
                class_idx = pred_idx

            score = logits[:, class_idx].sum()

        score.backward()

        if self.gradients is None:
            raise RuntimeError("Grad-CAM failed to capture gradients from the target layer.")

        activations = self.activations.detach()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam, pred_idx, probs

    def close(self):
        self.fwd_handle.remove()


def find_last_conv_layer(model):
    conv_layers = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    if not conv_layers:
        raise ValueError("No Conv2d layer found in model for Grad-CAM.")
    return conv_layers[-1]


def preprocess_image(image_path, image_size=(224, 224), grayscale=True):
    pil_mode = "L" if grayscale else "RGB"
    img = Image.open(image_path).convert(pil_mode)
    img = img.resize(image_size)

    if grayscale:
        img_np = np.array(img).astype(np.float32) / 255.0
        x = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0)
        display_img = np.stack([img_np, img_np, img_np], axis=-1)
    else:
        img_np = np.array(img).astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_norm = (img_np - mean) / std
        x = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0)
        display_img = img_np

    return display_img, x


def overlay_heatmap_on_image(image_rgb, cam, alpha=0.45):
    cmap = plt.get_cmap("jet")
    heatmap = cmap(cam)[..., :3]
    overlay = (1 - alpha) * image_rgb + alpha * heatmap
    overlay = np.clip(overlay, 0, 1)
    return heatmap, overlay


def load_model(model_path, device):
    model = CNNBinary()
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

In [ ]:
# ==== Example usage (edit these paths) ====
MODEL_PATH = "model_walls_clean.pth"
IMAGE_PATH = "Training_Pool_A_Walls/Cracked/7069-101.jpg"
CLASS_NAMES = ["Non-Cracked", "Cracked"]

# Optional: if your training used a different input size, change this.
IMAGE_SIZE = (256, 256)

# Safety switch: keep this False unless your CUDA setup is known to be stable.
USE_CUDA = False

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

if USE_CUDA and torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model = load_model(MODEL_PATH, device)
target_layer = find_last_conv_layer(model)

gradcam = GradCAM(model, target_layer)

In [ ]:

try:
    orig_img, x = preprocess_image(IMAGE_PATH, image_size=IMAGE_SIZE)
    x = x.to(device, non_blocking=True)

    cam, pred_idx, probs = gradcam(x)
    heatmap, overlay = overlay_heatmap_on_image(orig_img, cam, alpha=0.45)

    pred_name = CLASS_NAMES[pred_idx] if pred_idx < len(CLASS_NAMES) else f"Class {pred_idx}"
    conf = float(probs[pred_idx])

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(orig_img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap)
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(f"Overlay\nPrediction: {pred_name} ({conf:.2%})")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"Predicted class index: {pred_idx}")
    print(f"Class probabilities: {probs}")
finally:
    gradcam.close()